# Air quality and policy impact in India: fire smoke and GRAP
**BADIR notebook** · Parts 0 to 3 are built (setup, stakeholders stub, data, EDA). Parts 4 to 9 are owned by the teammates named in each heading.

Everything below starts from one file, `final_dataset.parquet` (54 stations x 14,136 IST hours, 70 columns). Results are *associations*, not proven causes.

| Part | BADIR | Owner |
|---|---|---|
| 0 Setup | D | MA |
| 1 Stakeholders and KPIs | B | MA |
| 2 Data sources and quality | D | RS, MA |
| 3 EDA | A | MA |
| 4 H1 fire and PM2.5 | A, I | MK |
| 5 H2 GRAP effect | A, I | RB |
| 6 Robustness, status, monitoring | I | MA, MK, RB |
| 7 Recommendations | R | RB |
| 8 Deployment demo | R | RS |
| 9 Final checks | all | all |

## Part 0 - Setup (BADIR: D)
Run this first. It works in Colab and on a local machine.

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
RANDOM_STATE = 42

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # EDIT: shared Drive folder that holds final_dataset.parquet and receives figures
    DRIVE_DIR = "/content/drive/MyDrive/air_quality_project"
    PARQUET = f"{DRIVE_DIR}/final_dataset.parquet"
    OUT_DIR = f"{DRIVE_DIR}/outputs"
else:
    DRIVE_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
    PARQUET = os.path.join(DRIVE_DIR, "data", "processed", "final_dataset.parquet")
    OUT_DIR = os.path.join(DRIVE_DIR, "outputs")

FIG_DIR = os.path.join(OUT_DIR, "figures")
TAB_DIR = os.path.join(OUT_DIR, "tables")
os.makedirs(FIG_DIR, exist_ok=True); os.makedirs(TAB_DIR, exist_ok=True)
print("Parquet:", PARQUET, "| exists:", os.path.exists(PARQUET))

In [ ]:
# Colab already has pandas, numpy, matplotlib, seaborn, scipy, statsmodels, pyarrow.
# RS: replace with `pip install -r requirements.txt` once the pinned file exists.
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
print("pandas", pd.__version__, "| python", sys.version.split()[0])

# One colour per group, fixed for every figure in the project (Okabe-Ito, colour-blind safe).
COL = {"treated": "#D55E00", "control": "#0072B2"}
GREY = "#999999"
sns.set_theme(style="whitegrid", font_scale=1.0)
plt.rcParams.update({"figure.dpi": 110, "axes.titleweight": "bold"})

def save_fig(fig, name):
    path = os.path.join(FIG_DIR, f"{name}.png")
    fig.savefig(path, dpi=200, bbox_inches="tight")
    print("saved", path)

def save_table(df, name, index=True):
    df.to_csv(os.path.join(TAB_DIR, f"{name}.csv"), index=index)

### 0.1 Load and validate the final dataset

In [ ]:
df = pd.read_parquet(PARQUET)
df["hour_ist"] = pd.to_datetime(df["hour_ist"])
df["date_ist"] = df["hour_ist"].dt.normalize()

n_st = df["station_id"].nunique()
n_hr = df["hour_ist"].nunique()
print("Rows:", len(df), "| columns:", df.shape[1], "| stations:", n_st, "| hours:", n_hr)

assert df.shape[1] == 70 + 1          # 70 columns in the file + date_ist added above
assert n_st == 54 and n_hr == 14136 and len(df) == 54 * 14136
assert df.groupby("group")["station_id"].nunique().to_dict() == {"control": 15, "treated": 39}
assert not df.duplicated(["station_id", "hour_ist"]).any()
assert df["pm25"].dropna().between(0, 1000).all()
print("Core validation checks passed (shape, 54 x 14,136, 39 treated + 15 control, unique keys, PM2.5 range).")
print("Window:", df["hour_ist"].min(), "to", df["hour_ist"].max())

## Part 1 - Stakeholders and KPIs (BADIR: B) - owner MA
> **TODO (MA):** fill from plan item M1, in the structure of the reference notebook: business problem, objective, key analytical questions, evaluation strategy, then an *Explanation* line.

Suggested layout: **Stakeholders** (table: who, decision they make, what they need from us) · **Business problem** · **Hypotheses H1 (fire smoke) and H2 (GRAP)** · **KPIs** (table: KPI, definition, unit, source column) · **Evaluation strategy** (split for H1, event windows for H2, fixed before looking at results).

## Part 2 - Data sources and quality (BADIR: D) - owners RS, MA
### 2.1 Sources and join flow
> **TODO (RS):** table T1 (source, provider, columns used, provenance from `docs/provenance_log.csv`) and figure F0 (flow with row counts from `docs/final_dataset_run_output.txt`).

### 2.2 Structure of the final table

In [ ]:
display(df.head())
display(df.dtypes.astype(str).value_counts().rename("columns").to_frame())
display(df.groupby(["group", "city"])["station_id"].nunique().rename("stations").to_frame())

### 2.3 Missingness (table T2) and the network outage

In [ ]:
key_cols = ["pm25","pm10","no2","o3","u10","wind_speed","blh_m","n_nh_100km_lag1","n_up_nh_100km_45deg_lag1"]
t2_cols = pd.DataFrame({"Missing_Count": df[key_cols].isna().sum(),
                        "Missing_Percentage": (df[key_cols].isna().mean()*100).round(2)})
display(t2_cols)

df["month_ist"] = df["hour_ist"].dt.to_period("M").astype(str)
t2_month = (df.assign(has_pm25=df["pm25"].notna())
              .pivot_table(index="group", columns="month_ist", values="has_pm25", aggfunc="mean")*100).round(1)
display(t2_month)
save_table(t2_cols, "T2_missingness_columns"); save_table(t2_month, "T2_missingness_by_month")

out_hours = df.loc[df["network_outage"], "hour_ist"].drop_duplicates().sort_values()
print(f"Network-outage hours: {len(out_hours)} of {n_hr} ({len(out_hours)/n_hr*100:.1f}%)")
runs = (out_hours.diff() != pd.Timedelta("1h")).cumsum()
outage_runs = out_hours.groupby(runs).agg(start="min", end="max", hours="size")
display(outage_runs)

**Interpretation (TODO MA, after reading the tables above):** state the PM2.5 coverage, which months are weak, and why the 11-20 Jan 2026 outage is *missing, not clean air*. State the rule: outage hours are excluded and a day counts only with at least 18 of 24 valid hours.

## Part 3 - Exploratory data analysis (BADIR: A, D) - owner MA
Frozen rule for every daily figure: a station-day is **valid** only if it has at least 18 of 24 valid hourly PM2.5 values (outage hours excluded).
### 3.1 Build the station-day table

In [ ]:
ok = df["pm25"].notna() & ~df["network_outage"]
sd = (df.assign(valid=ok, pm25_ok=df["pm25"].where(ok))
        .groupby(["station_id","group","city","date_ist"], observed=True)
        .agg(pm25=("pm25_ok","mean"), valid_hours=("valid","sum"))
        .reset_index())
sd.loc[sd["valid_hours"] < 18, "pm25"] = np.nan
print("Station-days:", len(sd), "| valid:", sd["pm25"].notna().sum(),
      f"({sd['pm25'].notna().mean()*100:.1f}%)")
display(sd.groupby("group")["pm25"].describe().round(1))

### 3.2 F1 - PM2.5 data availability

In [ ]:
avail = (df.assign(has=df["pm25"].notna())
           .pivot_table(index=["group","station_id"], columns="month_ist", values="has", aggfunc="mean"))
avail = avail.sort_index(level=["group","station_id"], ascending=[False, True])  # treated first
fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(avail.values, ax=ax, cmap="viridis", vmin=0, vmax=1,
            cbar_kws={"label": "Share of hours with PM2.5"},
            xticklabels=avail.columns, yticklabels=[f"{g[:1].upper()} {s}" for g, s in avail.index])
n_t = (avail.index.get_level_values(0) == "treated").sum()
ax.axhline(n_t, color="white", lw=3)
ax.set_title("F1. PM2.5 availability by station and month (T = treated above the line, C = control below)")
ax.set_xlabel("Month (IST)"); ax.set_ylabel("Station")
plt.xticks(rotation=60, ha="right"); plt.tight_layout()
save_fig(fig, "F1_pm25_availability"); plt.show()

**Interpretation (TODO MA):** what stands out (e.g. the Jan 2026 block), and one thing it does **not** show (presence of data, not data quality).

### 3.3 F2 - Station map

In [ ]:
st = df.drop_duplicates("station_id")[["station_id","city","group","latitude","longitude"]]
fig, ax = plt.subplots(figsize=(8, 6))
for g, sub in st.groupby("group"):
    ax.scatter(sub["longitude"], sub["latitude"], s=45, c=COL[g], label=f"{g} (n={len(sub)})",
               edgecolor="white", lw=0.6)
lat0, lon0 = 28.6139, 77.2090          # central Delhi (approximate); 25 km circle in degrees
th = np.linspace(0, 2*np.pi, 200)
ax.plot(lon0 + (25/(111.32*np.cos(np.radians(lat0))))*np.cos(th), lat0 + (25/110.57)*np.sin(th),
        color=GREY, ls="--", lw=1, label="25 km around central Delhi")
for c, sub in st.groupby("city"):
    ax.annotate(c, (sub["longitude"].mean(), sub["latitude"].mean()), xytext=(5, 8),
                textcoords="offset points", fontsize=9)
ax.set_xlabel("Longitude (degrees E)"); ax.set_ylabel("Latitude (degrees N)")
ax.set_title("F2. Monitoring stations (no basemap, coordinates in degrees)")
ax.legend(loc="lower right"); plt.tight_layout()
save_fig(fig, "F2_station_map"); plt.show()

**Interpretation (TODO MA).**

### 3.4 F3 - Daily PM2.5 and the GRAP timeline

In [ ]:
daily = (sd.groupby(["group","date_ist"])["pm25"]
           .agg(mean="mean", sd="std", n="count").reset_index())
daily["se"] = daily["sd"] / np.sqrt(daily["n"])
stage = df.groupby("date_ist")["grap_stage"].max()

fig, (ax, ax2) = plt.subplots(2, 1, figsize=(13, 6.5), sharex=True,
                              gridspec_kw={"height_ratios": [4, 1]})
for g in ["treated", "control"]:
    d = daily[daily["group"] == g]
    ax.plot(d["date_ist"], d["mean"], color=COL[g], lw=1.2, label=g)
    ax.fill_between(d["date_ist"], d["mean"]-1.96*d["se"], d["mean"]+1.96*d["se"], color=COL[g], alpha=0.2)
for s, e in zip(outage_runs["start"], outage_runs["end"]):
    for a in (ax, ax2):
        a.axvspan(s, e + pd.Timedelta("1h"), color=GREY, alpha=0.3, lw=0)
ax.axhline(60, color="k", ls=":", lw=1)
ax.text(daily["date_ist"].min(), 63, "60 ug/m3 24-h standard", fontsize=9)
ax.set_ylabel("Daily mean PM2.5 (ug/m3)"); ax.legend(title="Group")
ax.set_title("F3. Daily PM2.5 (mean, 95% band over stations) and GRAP stage; grey = network outage")
ax2.step(stage.index, stage.values, where="post", color="k")
ax2.set_ylabel("GRAP stage"); ax2.set_yticks(range(5)); ax2.set_xlabel("Date (IST)")
plt.tight_layout(); save_fig(fig, "F3_timeline"); plt.show()

**Interpretation (TODO MA):** when treated and control move together, when stages rise, and one limit: stage activation *follows* high pollution, so this plot cannot show that GRAP lowered it.

### 3.5 F4 - Seasonal profile

In [ ]:
sd["month"] = sd["date_ist"].dt.to_period("M").dt.to_timestamp()
seas = (sd.groupby(["group","month"])["pm25"]
          .agg(median="median", q1=lambda x: x.quantile(.25), q3=lambda x: x.quantile(.75), n="count")
          .reset_index())
fig, ax = plt.subplots(figsize=(12, 4.8))
for g in ["treated", "control"]:
    d = seas[seas["group"] == g]
    ax.plot(d["month"], d["median"], marker="o", color=COL[g], label=f"{g} median")
    ax.fill_between(d["month"], d["q1"], d["q3"], color=COL[g], alpha=0.2)
ax.set_ylabel("Station-day PM2.5 (ug/m3), median and IQR"); ax.set_xlabel("Month")
ax.set_title("F4. Monthly PM2.5 by group"); ax.legend(); plt.tight_layout()
save_fig(fig, "F4_seasonal"); plt.show()
display(seas.pivot(index="month", columns="group", values="n").add_prefix("n station-days "))

**Interpretation (TODO MA).**

### 3.6 F5 - Diurnal profile, winter vs monsoon

In [ ]:
dd = df[ok & df["season"].isin(["winter", "monsoon"])]
prof = dd.groupby(["group","season","hour"])["pm25"].agg(["mean","std","count"]).reset_index()
prof["ci"] = 1.96 * prof["std"] / np.sqrt(prof["count"])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, s in zip(axes, ["winter", "monsoon"]):
    for g in ["treated", "control"]:
        d = prof[(prof["group"] == g) & (prof["season"] == s)]
        ax.plot(d["hour"], d["mean"], color=COL[g], label=g)
        ax.fill_between(d["hour"], d["mean"]-d["ci"], d["mean"]+d["ci"], color=COL[g], alpha=0.2)
    ax.set_title(s.capitalize()); ax.set_xlabel("Hour of day (IST, start of hour)"); ax.set_xticks(range(0, 24, 3))
axes[0].set_ylabel("Mean PM2.5 (ug/m3)"); axes[0].legend()
fig.suptitle("F5. Diurnal PM2.5 profile with 95% CI", fontweight="bold"); plt.tight_layout()
save_fig(fig, "F5_diurnal"); plt.show()

**Interpretation (TODO MA).**

### 3.7 F6 - PM2.5 distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))
for g in ["treated", "control"]:
    v = np.sort(sd.loc[(sd["group"] == g) & sd["pm25"].notna(), "pm25"].values)
    v = v[v > 0]
    ax.plot(v, 1 - np.arange(1, len(v)+1) / len(v), color=COL[g], label=g)
ax.axvline(60, color="k", ls=":", lw=1); ax.text(62, 0.5, "60 ug/m3 (24-h standard)", rotation=90, fontsize=9)
ax.set_xscale("log"); ax.set_xlabel("Station-day PM2.5 (ug/m3, log axis)")
ax.set_ylabel("Share of station-days above x"); ax.set_title("F6. PM2.5 distribution (exceedance curve)")
ax.legend(); plt.tight_layout(); save_fig(fig, "F6_distribution"); plt.show()
exc = sd.dropna(subset=["pm25"]).groupby("group")["pm25"].apply(lambda x: (x > 60).mean()*100).round(1)
display(exc.rename("% of station-days above 60 ug/m3").to_frame())

**Interpretation (TODO MA).**

## Part 4 - H1: fire and PM2.5 (BADIR: A, I) - owner MK
> **TODO (MK):** lag correlation (F8, T3), regression (F9, T4), diagnostics (F10), prediction (F11), robustness.
> **Answer key to reproduce:** +11.8% PM2.5 per +10 upwind detections, 95% CI [-3.0, +28.9], 14 of 15 checks with the same sign.

## Part 5 - H2: GRAP effect (BADIR: A, I) - owner RB
> **TODO (RB):** event table (T5), counterfactual (F12, F13), placebo (F14), robustness (F15).
> **Answer key to reproduce:** 11 Nov 2025 event, +23% [-48, +190].

## Part 6 - Robustness, hypothesis status, monitoring (BADIR: I) - owners MA, MK, RB
> **TODO:** status table T7 following plan section 5.4; F16 monitoring (MA).

## Part 7 - Recommendations and roadmap (BADIR: R) - owner RB
> **TODO:** table T8. Every recommendation cites the evidence cell it rests on and states its limitation.

## Part 8 - Deployment demo (BADIR: R) - owner RS
> **TODO:** a demonstration (Colab links are temporary).

## Part 9 - Final checks - all
- [ ] Runtime > Restart and run all passes in a clean Colab session
- [ ] Every number in the report traces to a cell; answer-key values reproduce
- [ ] Every figure: units, source, interpretation sentence, one thing it does not show
- [ ] Wording is "associated with", never "caused"